In [58]:
import fitz
import os
import re
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document
from langchain_neo4j import Neo4jGraph

In [45]:
docs_name = [
    'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt',
    'Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt',
    'Datasets/Unstructured_Data/3okobat/3okobat.pdf',
    'Datasets/Unstructured_Data/Asle7a/asle7a.txt',
    'Datasets/Unstructured_Data/child/child.txt',
    'Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf',
    'Datasets/Unstructured_Data/drugs/drugs.txt',
    'Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf',
    'Datasets/Unstructured_Data/erhab/erhab.txt',
    'Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf',
    'Datasets/Unstructured_Data/technology/tech.txt'
    
]

In [46]:
base_url = "/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject"

def load_pdf_pymupdf(pdf_path):
    """Load PDF content using PyMuPDF"""
    try:
        doc = fitz.open(pdf_path)
        text = ""
        for page in doc:
            text += page.get_text()
        doc.close()
        return text
    except Exception as e:
        print(f"Error loading PDF {pdf_path}: {e}")
        return None

def load_text_file(txt_path):
    """Load text file content"""
    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            return f.read()
    except Exception as e:
        print(f"Error loading text file {txt_path}: {e}")
        return None

documents = []

for doc_name in docs_name:
    full_path = os.path.join(base_url, doc_name)
    ext = os.path.splitext(doc_name)[1].lower()

    content = None
    if ext == ".pdf":
        content = load_pdf_pymupdf(full_path)
    elif ext == ".txt":
        content = load_text_file(full_path)
    else:
        print(f"⚠ Skipping unsupported file type: {doc_name}")

    if content:
        documents.append(
            Document(
                page_content=content,
                metadata={"source": doc_name}
            )
        )
        print(f"✓ Successfully loaded: {doc_name}")
    else:
        print(f"✗ Failed to load: {doc_name}")

print(f"\nTotal loaded: {len(documents)}/{len(docs_name)} documents")


✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt
✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt
✓ Successfully loaded: Datasets/Unstructured_Data/3okobat/3okobat.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/Asle7a/asle7a.txt
✓ Successfully loaded: Datasets/Unstructured_Data/child/child.txt
✓ Successfully loaded: Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/drugs/drugs.txt
✓ Successfully loaded: Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/erhab/erhab.txt
✓ Successfully loaded: Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf
✓ Successfully loaded: Datasets/Unstructured_Data/technology/tech.txt

Total loaded: 11/11 documents


In [47]:
def extract_legal_articles(text, source):
    pattern = r'((?:المادة|مادة)\s*(?:\()?\s*\d+\s*(?:\))?)'
    parts = re.split(pattern, text)
    
    chunks = []
    current_chapter = None
    current_section = None
    
    for i in range(1, len(parts), 2):
        if i+1 >= len(parts):
            break
            
        article_header = parts[i].strip()
        article_content = parts[i+1].strip()
        
        if not article_content:
            continue
        
        num_match = re.search(r'\d+', article_header)
        article_num = int(num_match.group()) if num_match else None
        
        prev_text = parts[i-1] if i > 0 else ""
        chapter_match = re.search(r'الباب\s+([\w\s]+?)(?:\n|$)', prev_text[-500:])
        if chapter_match:
            current_chapter = chapter_match.group(1).strip()
        
        section_match = re.search(r'الفصل\s+([\w\s]+?)(?:\n|$)', prev_text[-500:])
        if section_match:
            current_section = section_match.group(1).strip()
        
        full_text = f"{article_header}\n{article_content}"
        
        chunks.append({
            "text": full_text,
            "metadata": {
                "source": source,
                "article_title": article_header,
                "article_number": article_num,
                "chapter": current_chapter,
                "section": current_section,
                "type": "legal_article"
            }
        })
    
    return chunks

In [48]:
all_legal_chunks = []

for doc in documents:
    chunks = extract_legal_articles(doc.page_content, doc.metadata["source"])
    all_legal_chunks.extend(chunks)
    print(f"📄 {doc.metadata['source']}: {len(chunks)} مادة")

print(f"\n✅ إجمالي: {len(all_legal_chunks)} مادة")

📄 Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt: 2 مادة
📄 Datasets/Unstructured_Data/3okobat/3okobat_num36_2020.txt: 1 مادة
📄 Datasets/Unstructured_Data/3okobat/3okobat.pdf: 512 مادة
📄 Datasets/Unstructured_Data/Asle7a/asle7a.txt: 94 مادة
📄 Datasets/Unstructured_Data/child/child.txt: 179 مادة
📄 Datasets/Unstructured_Data/dostor_gena2y/dostor.pdf: 166 مادة
📄 Datasets/Unstructured_Data/drugs/drugs.txt: 109 مادة
📄 Datasets/Unstructured_Data/egra2at_gena2ya/egra2at_gena2ya.pdf: 59 مادة
📄 Datasets/Unstructured_Data/erhab/erhab.txt: 67 مادة
📄 Datasets/Unstructured_Data/na2d/mbade2_ma7kmt_elna2d.pdf: 305 مادة
📄 Datasets/Unstructured_Data/technology/tech.txt: 53 مادة

✅ إجمالي: 1547 مادة


In [49]:
long_articles = [c for c in all_legal_chunks if len(c["text"]) > 2000]
print(f"⚠️ {len(long_articles)} long_articles")

⚠️ 98 long_articles


In [50]:
all_legal_chunks

[{'text': 'المادة ٣٩٢\nمن قانون العقوبات الصادر بالقانون رقم ٥٨ لسنة 1937 النص الآتي:',
  'metadata': {'source': 'Datasets/Unstructured_Data/3okobat/3okobat_num6_2020.txt',
   'article_title': 'المادة ٣٩٢',
   'article_number': 392,
   'chapter': None,
   'section': None,
   'type': 'legal_article'}},
 {'text': 'مادة ٣٩٢\nكل من صدر عليه حكم قضائي واجب النفاذ بدفع نفقة لزوجه أو أقاربه أو أصهاره أو أجرة حضانة أو رضاعة أو مسكن وامتنع عن الدفع مع قدرته عليه لمدة ثلاثة أشهر بعد التنبيه عليه بالدفع، يعاقب بالحبس مدة لا تزيد على سنة وبغرامة لا تتجاوز خمسة آلاف جنيه أو بإحدى هاتين العقوبتين، ولا ترفع الدعوى عليه إلا بناء على شكوى أو طلب من صاحب الشأن، وإذا رفعت بعد الحكم عليه دعوى ثانية عن هذه الجريمة فتكون عقوبته الحبس مدة لا تزيد على سنة، ويترتب على الحكم الصادر بالإدانة تعليق استفادة المحكوم عليه من الخدمات المطلوب الحصول عليها بمناسبة ممارسته نشاطه المهني والتي تقدمها الجهات الحكومية والهيئات العامة ووحدات القطاع العام وقطاع الأعمال العام والجهات التي تؤدي خدمات مرافق عامة، حتى أداء ما تجم

In [ ]:
documents = []

for chunk in all_legal_chunks:
    doc = Document(
        page_content=chunk["text"],
        metadata=chunk.get("metadata", {})
    )
    documents.append(doc)

print(f"✅ تم تحويل {len(documents)} chunks إلى Document objects")

✅ تم تحويل 1547 chunks إلى Document objects


In [52]:
os.chdir("/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject")
os.getcwd()

'/media/mohamed-gaber/01D96E17175F8760/Artifitial_intelligence/GraduationProject'

In [53]:
from src.LLMs.Qwen_Model import ArabicModel

print("Loading LLM...")
llm_ar = ArabicModel()
llm = llm_ar.get_ar_llm()
print("LLM Ready...")

Loading LLM...
LLM Ready...


In [54]:
llm_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=[
        "مادة_قانونية",  # Legal Article
        "جريمة",         # Crime
        "عقوبة",         # Penalty
        "شرط",           # Condition
        "باب",           # Chapter
        "فصل"            # Section
    ],
    allowed_relationships=[
        "يُعاقب_بـ",     # is_punished_by: جريمة → عقوبة
        "يشترط",         # requires: مادة_قانونية → شرط
        "يُحيل_إلى",     # refers_to: مادة_قانونية → مادة_قانونية
        "ينتمي_لـ",      # belongs_to: مادة_قانونية → باب/فصل
        "يُشدد_بـ",      # aggravated_by: عقوبة → شرط
        "يُخفف_بـ"       # mitigated_by: عقوبة → شرط
    ]
)

In [60]:
graph_documents = llm_transformer.convert_to_graph_documents(documents)

BadRequestError: Error code: 400 - {'error': {'message': "Failed to call a function. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': '{"name": "DynamicGraph", "arguments": {"nodes": [{"id": "مادة 83", "type": "مادة_قانونية"}], "relationships": []}}'}}

In [59]:
from src.Config.config import NEO4J_URI , NEO4J_PASSWORD , NEO4J_USERNAME

try:
    graph_db = Neo4jGraph(
        url=NEO4J_URI,
        username=NEO4J_USERNAME,
        password=NEO4J_PASSWORD 
    )
    
    print("⏳ حفظ الـ graph في Neo4j...")
    graph_db.add_graph_documents(graph_documents)
    print("✅ تم حفظ الـ graph في Neo4j")
    
except Exception as e:
    print(f"⚠️ لم يتم الاتصال بـ Neo4j: {e}")
    print("يمكنك تخطي هذه الخطوة إذا لم يكن Neo4j مثبت")

⏳ حفظ الـ graph في Neo4j...
✅ تم حفظ الـ graph في Neo4j
